# **처음 본 숫자 맞추기(Many-to-One)**

* RNN은 이전 데이터를 기억할 수 있는 능력이 있습니다.
* 따라서 순서가 중요한 시퀀스데이터 처리에 사용되는 고전적인 모델입니다.
* 우리는 맨 처음 본 숫자를 맞추는 RNN모델을 만들어보려고 합니다.

## **1. 데이터 준비**

- T,B,F (20, 32, 1) 크기의 랜덤 float형 데이터 (x)를 만들어 주세요.
- 데이터는 0부터 9사이의 숫자가 됩니다.
- 정답 (y)는 x의 0번째 (맨 처음 숫자)로 정의합니다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [41]:
# code here
T = 5      
B = 32

In [42]:
def make_data():
    X = torch.randint(0, 10, (T, B, 1)).float()
    y = X[0]        # 첫번째 숫자가 정답
    return X, y

## **2. 모델 만들기**

레이어 구성은 다음과 같습니다.
- rnn layer(1,32) -> Linear layer(32,1)
- linear layer에 들어가는 정보는 rnn에서 나온 마지막 정보만 넣습니다.

In [51]:
# code here
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32)
        self.linear = nn.Linear(32, 1)  

    def forward(self, X):
        h, _ = self.rnn(X)
        return self.linear(h[-1])      # h[-1] : 마지막 기억

In [52]:
model = RNN()
print(model)

RNN(
  (rnn): RNN(1, 32)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


## **3. 손실함수와 옵티마이저**

**Q1. 아래와 같이 옵티마이저를 설정합니다.**

- Optimizer : Adam
- learning rate : 0.01
- 손실 함수는 forward 내에서 F.mse_loss를 활용합니다.

In [53]:
# code here
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [12]:
criterion = nn.MSELoss()

## **4. 학습**


**Q1. 학습 모드로, 생성된 rnn 신경망을 학습시킵니다.**
- epochs : 1000
- 100번의 epoch마다 loss를 출력해보도록 합시다.


In [55]:
# code here
epochs = 1000

for epoch in range(epochs):
    model.train()
    X, y = make_data()
    y_pred = model(X)             # 순전파
    loss = criterion(y_pred, y)   # loss
    optimizer.zero_grad() 
    loss.backward()               # 역전파
    optimizer.step()        

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | loss {loss}")

epoch 100 | loss 7.280136585235596
epoch 200 | loss 9.412108421325684
epoch 300 | loss 10.031733512878418
epoch 400 | loss 10.562605857849121
epoch 500 | loss 10.691980361938477
epoch 600 | loss 7.682956218719482
epoch 700 | loss 0.7908248901367188
epoch 800 | loss 0.3656763732433319
epoch 900 | loss 0.08257023990154266
epoch 1000 | loss 0.0037363250739872456


## **5. 평가**

**Q1. 평가 모드로, 학습된 신경망의 정확도를 구합니다.**
- 총 20번 새로운 데이터를 만들어 예측을 시키고, 결과를 구해 봅시다.


In [60]:
# code here

test_epoch = 20
cnt = 0
total = 0
for epoch in range(test_epoch):
    model.eval()
    X, y = make_data()
    with torch.no_grad():
        y_pred = model(X)
        for b in range(B):
            pred = y_pred[b][0].round().item()
            # 실제 결과랑 비교 해주기
            ans = y[b][0]
            if pred == ans:
                cnt  += 1           # 하나의 정답을 더 맞췄다.
            total += 1

print(f"accuracy : {cnt / total}")

accuracy : 0.8859375


## **6. timestep 바꾸기**

**Q1. timestep=7일때의 정확도를 평가해보자.**

In [59]:
# code here
T = 7

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

epochs = 10000

for epoch in range(epochs):
    model.train()
    X, y = make_data()
    y_pred = model(X)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | loss {loss}")


epoch 100 | loss 10.029109954833984
epoch 200 | loss 7.167999267578125
epoch 300 | loss 6.983336448669434
epoch 400 | loss 11.151617050170898
epoch 500 | loss 9.927484512329102
epoch 600 | loss 7.15672492980957
epoch 700 | loss 7.457887172698975
epoch 800 | loss 6.533757209777832
epoch 900 | loss 6.669986724853516
epoch 1000 | loss 0.08876433968544006
epoch 1100 | loss 0.0063697961159050465
epoch 1200 | loss 0.002756949048489332
epoch 1300 | loss 0.000359442550688982
epoch 1400 | loss 0.0008742097415961325
epoch 1500 | loss 0.0006630881689488888
epoch 1600 | loss 0.00034228124422952533
epoch 1700 | loss 0.0004265758325345814
epoch 1800 | loss 0.000140504926093854
epoch 1900 | loss 0.0005147436750121415
epoch 2000 | loss 0.0005259005120024085
epoch 2100 | loss 0.001065061311237514
epoch 2200 | loss 0.0034021802712231874
epoch 2300 | loss 0.0023254125844687223
epoch 2400 | loss 0.0006088005611672997
epoch 2500 | loss 0.0003512479888740927
epoch 2600 | loss 9.686215344117954e-05
epoch 270

In [ ]:
# code here

test_epoch = 20
cnt = 0
total = 0
for epoch in range(test_epoch):
    model.eval()
    X, y = make_data()
    with torch.no_grad():
        y_pred = model(X)
        for b in range(B):
            pred = y_pred[b][0].round().item()
            # 실제 결과랑 비교 해주기
            ans = y[b][0]
            if pred == ans:
                cnt  += 1           # 하나의 정답을 더 맞췄다.
            total += 1

print(f"accuracy : {cnt / total}")

accuracy : 0.8484375


**Q2. timestep=10일때의 정확도를 평가해보자.**

In [ ]:
# code here
T = 10

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

epochs = 2000

for epoch in range(epochs):
    model.train()
    X, y = make_data()
    y_pred = model(X)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | loss {loss}")



epoch 100 | loss 6.998044013977051
epoch 200 | loss 7.479468822479248
epoch 300 | loss 6.756628513336182
epoch 400 | loss 6.05042839050293
epoch 500 | loss 7.3631439208984375
epoch 600 | loss 9.025697708129883
epoch 700 | loss 7.005327224731445
epoch 800 | loss 8.065807342529297
epoch 900 | loss 9.675346374511719
epoch 1000 | loss 10.15485954284668
epoch 1100 | loss 7.641287326812744
epoch 1200 | loss 7.519286632537842
epoch 1300 | loss 6.874434471130371
epoch 1400 | loss 6.487287998199463
epoch 1500 | loss 5.40033483505249
epoch 1600 | loss 4.199917316436768
epoch 1700 | loss 6.531885623931885
epoch 1800 | loss 7.427842140197754
epoch 1900 | loss 5.329858779907227
epoch 2000 | loss 7.0171427726745605


In [ ]:
# code here

test_epoch = 20
cnt = 0
total = 0
for epoch in range(test_epoch):
    model.eval()
    X, y = make_data()
    with torch.no_grad():
        y_pred = model(X)
        for b in range(B):
            pred = y_pred[b][0].round().item()
            # 실제 결과랑 비교 해주기
            ans = y[b][0]
            if pred == ans:
                cnt  += 1           # 하나의 정답을 더 맞췄다.
            total += 1

print(f"accuracy : {cnt / total}")

accuracy : 0.14375


**Q3. timestep=20일때의 정확도를 평가해보자.**

In [62]:
# code here
# code here
T = 20

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

epochs = 2000

for epoch in range(epochs):
    model.train()
    X, y = make_data()
    y_pred = model(X)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | loss {loss}")



epoch 100 | loss 9.115096092224121
epoch 200 | loss 8.557634353637695
epoch 300 | loss 9.606391906738281
epoch 400 | loss 10.596479415893555
epoch 500 | loss 7.449306488037109
epoch 600 | loss 6.011029243469238
epoch 700 | loss 5.215617656707764
epoch 800 | loss 6.383457183837891
epoch 900 | loss 8.472955703735352
epoch 1000 | loss 7.246452331542969
epoch 1100 | loss 9.314245223999023
epoch 1200 | loss 7.936757564544678
epoch 1300 | loss 7.561418533325195
epoch 1400 | loss 5.617501735687256
epoch 1500 | loss 6.953366756439209
epoch 1600 | loss 8.43781566619873
epoch 1700 | loss 7.790570259094238
epoch 1800 | loss 8.529994010925293
epoch 1900 | loss 8.684372901916504
epoch 2000 | loss 9.215737342834473


In [63]:
# code here

test_epoch = 20
cnt = 0
total = 0
for epoch in range(test_epoch):
    model.eval()
    X, y = make_data()
    with torch.no_grad():
        y_pred = model(X)
        for b in range(B):
            pred = y_pred[b][0].round().item()
            # 실제 결과랑 비교 해주기
            ans = y[b][0]
            if pred == ans:
                cnt  += 1           # 하나의 정답을 더 맞췄다.
            total += 1

print(f"accuracy : {cnt / total}")

accuracy : 0.1


## **LSTM VS RNN 비교**

In [2]:
def make_data():
    X = torch.randint(0, 10, (T, B, 1)).float()
    y = X[0]        # 첫번째 숫자가 정답
    return X, y

In [3]:
T = 20
B = 32

In [4]:
# code here
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32)
        self.linear = nn.Linear(32, 1)  

    def forward(self, X):
        h, _ = self.lstm(X)
        return self.linear(h[-1])      # h[-1] : 마지막 기억

In [5]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32)
        self.linear = nn.Linear(32, 1)  

    def forward(self, X):
        h, _ = self.rnn(X)
        return self.linear(h[-1])      # h[-1] : 마지막 기억

In [6]:
model_lstm = LSTM()

In [7]:
model_rnn = RNN()

In [8]:
criterion = nn.MSELoss()

In [9]:
optimizer_lstm = torch.optim.Adam(model_lstm.parameters(), lr = 0.01)

In [10]:
optimizer_rnn = torch.optim.Adam(model_rnn.parameters(), lr = 0.01)

In [11]:
# code here
epochs = 3000

for epoch in range(epochs):
    model_lstm.train()
    X, y = make_data()
    y_pred_lstm = model_lstm(X)
    loss_lstm = criterion(y_pred_lstm, y)
    optimizer_lstm.zero_grad()
    loss_lstm.backward()
    optimizer_lstm.step()

    model_rnn.train()
    y_pred_rnn = model_rnn(X)
    loss_rnn = criterion(y_pred_rnn, y)
    optimizer_rnn.zero_grad()
    loss_rnn.backward()
    optimizer_rnn.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1} | LSTM loss: {loss_lstm} | RNN loss: {loss_rnn}")

epoch 100 | LSTM loss: 5.797316074371338 | RNN loss: 5.786901473999023
epoch 200 | LSTM loss: 8.34898567199707 | RNN loss: 8.492639541625977
epoch 300 | LSTM loss: 11.063685417175293 | RNN loss: 11.059578895568848
epoch 400 | LSTM loss: 9.29266357421875 | RNN loss: 9.458641052246094
epoch 500 | LSTM loss: 8.5317964553833 | RNN loss: 8.5258150100708
epoch 600 | LSTM loss: 7.921511650085449 | RNN loss: 7.884757041931152
epoch 700 | LSTM loss: 10.731966018676758 | RNN loss: 10.479169845581055
epoch 800 | LSTM loss: 7.391097545623779 | RNN loss: 7.638533115386963
epoch 900 | LSTM loss: 9.642338752746582 | RNN loss: 9.656421661376953
epoch 1000 | LSTM loss: 7.735112190246582 | RNN loss: 7.731273651123047
epoch 1100 | LSTM loss: 7.277468681335449 | RNN loss: 7.32505989074707
epoch 1200 | LSTM loss: 9.232858657836914 | RNN loss: 9.209856033325195
epoch 1300 | LSTM loss: 9.05624008178711 | RNN loss: 9.045382499694824
epoch 1400 | LSTM loss: 8.83649730682373 | RNN loss: 8.820140838623047
epoch 